In [18]:
import pandas as pd

INPUT_CSV = './resources/cnmf_factor_cluster_top_genes_200.csv'
df = pd.read_csv(INPUT_CSV)
df.head()

,Cluster 0 genes,Cluster 0 # of shared factors,Cluster 1 genes,Cluster 1 # of shared factors,Cluster 2 genes,Cluster 2 # of shared factors,Cluster 3 genes,Cluster 3 # of shared factors,Cluster 4 genes,Cluster 4 # of shared factors,...,Cluster 30 genes,Cluster 30 # of shared factors,Cluster 31 genes,Cluster 31 # of shared factors,Cluster 32 genes,Cluster 32 # of shared factors,Cluster 33 genes,Cluster 33 # of shared factors,Cluster 34 genes,Cluster 34 # of shared factors
0,FAM189A2,16,TNC,14,DPP6,12,CFI,3,ALCAM,13,...,ABCA1,2,ABLIM3,1,WDR62,34,HSPH1,6,ATP13A4,5
1,OGFRL1,14,CD44,14,CHST11,11,SNTG1,3,GALNT13,12,...,FNDC3B,2,NHS,1,C21ORF58,34,PPP1R15A,6,LINC00299,5
2,MAP3K5,14,VCL,13,MAP3K1,11,PLEKHG1,3,DNM3,12,...,PKP4,2,MXD1,1,MSH5,34,UBC,6,AQP4,5
3,ITPR2,14,SAMD4A,13,SLC24A3,11,PRKD1,3,ADGRL3,12,...,PLCB1,2,MYOF,1,SMC4,34,CCDC59,6,HIF3A,5
4,ETNPPL,14,IGFBP7,13,NRXN1,11,ETV6,3,MIR181A2HG,12,...,FAM20C,2,NAMPT,1,LMNB1,34,UBB,6,LINC01727,5


In [19]:
records = dict()
for c in df.columns:
    if 'genes' not in c.lower():
        continue
    gene_list = list(df[c].dropna())  # avoid NaNs
    records['cluster'] = {
    'plain_query': (
        f"What might the following enriched gene list say about the type: {gene_list}"
    ),
    'contextual_query': (
        "The following gene list represents a gene program found in some subset of malignant cells isolated\n"
        "from an IDH-mutant astrocytoma. Please make predictions of how this gene program affects the malignant cells\n"
        "that express it — including structure, function (biological processes), metabolic state, interactions with the\n"
        "ECM and other cells. Use evidence not only from the astrocytoma literature and other relevant cancer literature,\n"
        "but also from the normal development and function of astrocytes.\n"
        "Rank predictions more highly where multiple genes in the list are known to be involved in a relevant process.\n"
        "Where multiple genes are known to be required for that process, assess whether all required genes are present and rank higher if they are.\n"
        + "Gene List: " + str(gene_list)
    )
    }


In [ ]:
import nest_asyncio
nest_asyncio.apply()

from futurehouse_client import FutureHouseClient, JobNames
from pathlib import Path
from aviary.core import DummyEnv
from dotenv import load_dotenv
import os
import json
import copy
import asyncio

load_dotenv(dotenv_path='../../.env')

client = FutureHouseClient(
    api_key=os.getenv("FHK")
)

query_types = {
    'queries': ['plain_query', 'contextual_query'],
    'Jobs': [
        {'name': 'crow', 'job': JobNames.CROW},
        {'name': 'falcon', 'job': JobNames.FALCON}
    ]
}
jout = {}

async def run_task_and_save(cluster, r, q, j, progress):
    task_data = {
        "name": j['job'],
        "query": r[q],
    }
    print(f"Submitting: cluster={cluster}, query={q}, job={j['name']}")
    task_response = await client.run_tasks_until_done_async(task_data)
    path = Path(f"output/{j['name']}/{q}")
    path.mkdir(parents=True, exist_ok=True)
    js = json.loads(task_response[0].model_dump_json())
    out = f"## Cluster: {cluster}\n\n"
    out += f"### Result: \n\n {js['formatted_answer']}"
    with open(f"{path}/{cluster}.md", 'w') as f:
        f.write(out)
    jout.setdefault(cluster, copy.deepcopy(r))
    jout[cluster][j['name']] = copy.deepcopy(js)
    progress['done'] += 1
    print(f"Finished: cluster={cluster}, query={q}, job={j['name']} ({progress['done']}/{progress['total']})")

async def main():
    tasks = []
    progress = {'done': 0, 'total': 0}
    for cluster, r in records.items():
        for q in query_types['queries']:
            for j in query_types['Jobs']:
                progress['total'] += 1
                tasks.append(run_task_and_save(cluster, r, q, j, progress))
    semaphore = asyncio.Semaphore(5)  # Adjust concurrency as needed

    async def sem_task(task):
        async with semaphore:
            await task

    await asyncio.gather(*(sem_task(t) for t in tasks))

await main()

with open("FutureHouse_output.json", "w") as f:
    json.dump(jout, f, indent=4)
print("Finished")

{'name': <JobNames.FALCON: 'job-futurehouse-paperqa2-deep'>, 'query': "What might the following enriched gene list say about the type: ['ATP13A4', 'LINC00299', 'AQP4', 'HIF3A', 'LINC01727', 'SLC6A1', 'LDB2', 'ATP1A2', 'SLC4A4', 'SLC1A3', 'LRIG1', 'LIFR', 'SHROOM3', 'AHCYL2', 'C1ORF61', 'DTNA', 'ITPR2', 'RGMA', 'TFCP2L1', 'COL23A1', 'ALDH1L1', 'PCDH9', 'ADGRV1', 'KCNJ16', 'KCNN3', 'PDZRN3', 'KLHDC8A', 'MEOX2', 'AL589740.1', 'FMN2', 'SLC6A11', 'SLC1A2', 'SLC15A2', 'ESRRG', 'CADM1', 'CARMIL1', 'SEC14L1', 'CHPT1', 'DGKG', 'DCLK2', 'NPAS3', 'RORA', 'ZNF521', 'AC008957.1', 'MAPK4', 'WIPF3', 'MEIS1', 'AC008957.2', 'BBOX1', 'GLI3', 'PPARGC1A', 'AQP4-AS1', 'NCAN', 'RANBP3L', 'DMD', 'PLD5', 'AC012405.1', 'LRRC3B', 'GPM6A', 'RGS20', 'ALK', 'PHYHIPL', 'MAP3K5', 'PTPRJ', 'ARHGAP26', 'EDNRB', 'C1ORF21', 'TRPS1', 'TTYH3', 'FAT3', 'PREX2', 'PTPRT', 'ELOVL2', 'TTYH1', 'FGD4', 'ARHGEF26', 'AC024145.1', 'CACHD1', 'SLCO1C1', 'PXDN', 'GRIA1', 'LSAMP', 'PPP2R2B', 'AL591686.2', 'GRIA4', 'RGS7', 'HTRA1', 'TMT

In [ ]:
print(str(JobNames.CROW))


job-futurehouse-paperqa2
